In [6]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from IPython.display import display, HTML
import contextlib, io, warnings
warnings.filterwarnings("ignore")


def volume_signal(
    ticker,
    start        = "2016-01-01",
    obv_window   = 20,
    cmf_window   = 20,
    vol_short    = 20,
    vol_long     = 50,
    vwap_window  = 20,
    plot         = True,
):
    """
    VolumeSignal — Volume-based market conviction analyzer.
    Visual identity: warm finance theme (cream, gold, burgundy).

    Parameters
    ----------
    ticker       : str   — required. Any yfinance ticker e.g. "ABUK.CA", "AAPL"
    start        : str   — optional. Historical start date (default "2016-01-01")
    obv_window   : int   — optional. OBV signal line smoothing window (default 20)
    cmf_window   : int   — optional. Chaikin Money Flow window (default 20)
    vol_short    : int   — optional. Short volume MA window (default 20)
    vol_long     : int   — optional. Long volume MA window (default 50)
    vwap_window  : int   — optional. Rolling VWAP window (default 20)
    plot         : bool  — optional. Show dashboard (default True)
    """

    # ── Warm finance color palette ───────────────
    C = {
        "bg":          "#FAF6F0",   # warm cream background
        "bg_dark":     "#F0E8DC",   # slightly darker cream
        "gold":        "#B8860B",   # dark goldenrod
        "gold_light":  "#F5D78E",   # light gold
        "gold_pale":   "#FDF3D0",   # very pale gold
        "burgundy":    "#6B1D2A",   # deep burgundy
        "burgundy_lt": "#F5E0E3",   # pale burgundy
        "brown":       "#3E2C1A",   # dark brown (text)
        "brown_mid":   "#7A5C3A",   # medium brown
        "brown_lt":    "#C4A882",   # light brown
        "rust":        "#B5451B",   # rust/terra cotta
        "rust_lt":     "#FAEAE3",   # pale rust
        "sage":        "#5C7A5A",   # muted sage green
        "sage_lt":     "#E8F0E8",   # pale sage
        "ivory":       "#FFFFF0",   # ivory white
        "border":      "#D4C4A8",   # warm border
    }

    # ─────────────────────────────────────────────
    # 1. Download & validate (silent)
    # ─────────────────────────────────────────────
    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()):
        raw = yf.download(ticker, start=start, progress=False)

    required_cols = ["Close", "High", "Low", "Volume"]
    for col in required_cols:
        if col not in raw.columns:
            raise ValueError(f"Missing column '{col}' for '{ticker}'.")

    df = raw[required_cols].copy().dropna()

    if df.empty:
        raise ValueError(f"No data returned for '{ticker}'.")
    if len(df) < vol_long + 10:
        raise ValueError(f"Not enough history for '{ticker}'.")

    close  = df["Close"].squeeze()
    high   = df["High"].squeeze()
    low    = df["Low"].squeeze()
    volume = df["Volume"].squeeze()

    # ─────────────────────────────────────────────
    # 2. Compute indicators
    # ─────────────────────────────────────────────

    # OBV
    price_dir  = np.sign(close.diff().fillna(0))
    obv        = (price_dir * volume).cumsum()
    obv_signal = obv.rolling(obv_window).mean()

    # CLV → Money Flow Volume → A/D Line
    hl_range          = (high - low).replace(0, np.nan)
    clv               = ((close - low) - (high - close)) / hl_range
    clv               = clv.replace([np.inf, -np.inf], 0).fillna(0)
    money_flow_volume = clv * volume
    ad_line           = money_flow_volume.cumsum()

    # CMF
    cmf = (
        money_flow_volume.rolling(cmf_window).sum()
        / volume.rolling(cmf_window).sum()
    )

    # Volume MAs & ratio
    vol_ma_short = volume.rolling(vol_short).mean()
    vol_ma_long  = volume.rolling(vol_long).mean()
    volume_ratio = vol_ma_short / vol_ma_long

    # Rolling VWAP & long-run VWAP
    rolling_vwap  = (
        (close * volume).rolling(vwap_window).sum()
        / volume.rolling(vwap_window).sum()
    )
    long_run_vwap = (close * volume).cumsum() / volume.cumsum()

    # Confirmation score components
    price_up      = close.diff() > 0
    price_down    = close.diff() < 0
    vol_above_avg = volume > vol_ma_short
    confirmed_up   = (price_up   & vol_above_avg).rolling(20).mean()
    confirmed_down = (price_down & vol_above_avg).rolling(20).mean()
    weak_moves     = (~vol_above_avg).rolling(20).mean()

    # ─────────────────────────────────────────────
    # 3. Extract current readings
    # ─────────────────────────────────────────────
    latest_close     = float(close.iloc[-1])
    latest_obv       = float(obv.iloc[-1])
    latest_obv_sig   = float(obv_signal.dropna().iloc[-1])
    latest_cmf       = float(cmf.dropna().iloc[-1])
    latest_vol_ratio = float(volume_ratio.dropna().iloc[-1])
    latest_vwap      = float(rolling_vwap.dropna().iloc[-1])
    latest_lr_vwap   = float(long_run_vwap.iloc[-1])
    latest_conf_up   = float(confirmed_up.dropna().iloc[-1])
    latest_conf_down = float(confirmed_down.dropna().iloc[-1])
    latest_weak      = float(weak_moves.dropna().iloc[-1])

    obv_recent = float(obv.iloc[-5:].mean())
    obv_prior  = float(obv.iloc[-10:-5].mean())
    obv_trend  = "Rising ↑" if obv_recent > obv_prior else "Falling ↓"

    ad_recent  = float(ad_line.iloc[-5:].mean())
    ad_prior   = float(ad_line.iloc[-10:-5].mean())
    ad_trend   = "Accumulation ↑" if ad_recent > ad_prior else "Distribution ↓"

    vwap_pct   = (latest_close - latest_vwap) / latest_vwap
    vwap_sig   = "Above VWAP ↑" if latest_close > latest_vwap else "Below VWAP ↓"

    if latest_vol_ratio > 1.15:   vol_activity = "High ↑↑"
    elif latest_vol_ratio > 1.0:  vol_activity = "Above average ↑"
    elif latest_vol_ratio > 0.85: vol_activity = "Below average ↓"
    else:                         vol_activity = "Low ↓↓"

    if latest_cmf > 0.15:         cmf_label = "Strong accumulation"
    elif latest_cmf > 0.0:        cmf_label = "Mild accumulation"
    elif latest_cmf > -0.15:      cmf_label = "Mild distribution"
    else:                         cmf_label = "Strong distribution"

    # ─────────────────────────────────────────────
    # 4. Conviction score (0–100)
    # ─────────────────────────────────────────────
    score = 0
    if obv_recent > obv_prior:          score += 20
    if latest_cmf > 0.15:               score += 25
    elif latest_cmf > 0.0:              score += 12
    if latest_vol_ratio > 1.15:         score += 15
    elif latest_vol_ratio > 1.0:        score += 8
    if latest_close > latest_vwap:      score += 20
    if ad_recent > ad_prior:            score += 10
    if latest_conf_up > latest_conf_down: score += 10

    if score >= 75:
        conviction = "STRONG BULLISH"
        conv_color = C["sage"];      conv_bg = C["sage_lt"]
        verdict    = "✦ Volume strongly confirms bullish trend"
    elif score >= 50:
        conviction = "MILD BULLISH"
        conv_color = C["gold"];      conv_bg = C["gold_pale"]
        verdict    = "◈ Volume mildly supportive of upside"
    elif score >= 35:
        conviction = "NEUTRAL"
        conv_color = C["brown_mid"]; conv_bg = C["bg_dark"]
        verdict    = "◇ Mixed signals — no clear volume conviction"
    elif score >= 20:
        conviction = "MILD BEARISH"
        conv_color = C["rust"];      conv_bg = C["rust_lt"]
        verdict    = "◈ Volume mildly contradicts bullish thesis"
    else:
        conviction = "STRONG BEARISH"
        conv_color = C["burgundy"];  conv_bg = C["burgundy_lt"]
        verdict    = "✦ Volume strongly contradicts any bullish thesis"

    # ─────────────────────────────────────────────
    # 5. HTML summary table — warm finance style
    # ─────────────────────────────────────────────
    def pill(text, tone="neutral"):
        styles = {
            "bull":    f"background:{C['sage_lt']};   color:{C['sage']};",
            "bear":    f"background:{C['burgundy_lt']};color:{C['burgundy']};",
            "neutral": f"background:{C['gold_pale']};  color:{C['gold']};",
            "warn":    f"background:{C['rust_lt']};    color:{C['rust']};",
        }
        s = styles.get(tone, styles["neutral"])
        return (f'<span style="{s} font-weight:600; padding:2px 10px;'
                f'border-radius:4px; font-size:11px; '
                f'letter-spacing:0.03em;">{text}</span>')

    def tone(positive, strong=False, neutral=False):
        if neutral: return "neutral"
        if positive: return "bull"
        return "bear" if strong else "warn"

    rows = [
        ("Current Price",
         f"{latest_close:,.2f}",
         pill(f"vs VWAP {vwap_pct:+.2%}",
              tone(latest_close > latest_vwap))),

        ("On-Balance Volume",
         f"{latest_obv:,.0f}",
         pill(obv_trend, tone("↑" in obv_trend))),

        ("Chaikin Money Flow",
         f"{latest_cmf:+.4f}",
         pill(cmf_label,
              "neutral" if abs(latest_cmf) < 0.05
              else tone(latest_cmf > 0, strong=abs(latest_cmf) > 0.15))),

        ("A/D Line Trend",
         "Accumulating" if ad_recent > ad_prior else "Distributing",
         pill(ad_trend, tone(ad_recent > ad_prior))),

        ("Volume Activity",
         f"Ratio {latest_vol_ratio:.2f}×",
         pill(vol_activity,
              "neutral" if abs(latest_vol_ratio - 1) < 0.15
              else tone(latest_vol_ratio >= 1.0))),

        ("Rolling VWAP",
         f"{latest_vwap:,.2f}",
         pill(vwap_sig, tone(latest_close > latest_vwap))),

        ("Long-Run VWAP",
         f"{latest_lr_vwap:,.2f}",
         pill("Above ↑" if latest_close > latest_lr_vwap else "Below ↓",
              tone(latest_close > latest_lr_vwap))),

        ("Confirmed Up Days (20D)",
         f"{latest_conf_up:.1%}",
         pill("Strong buying" if latest_conf_up > 0.3 else "Weak buying",
              tone(latest_conf_up > latest_conf_down))),

        ("Confirmed Down Days (20D)",
         f"{latest_conf_down:.1%}",
         pill("Heavy selling" if latest_conf_down > 0.3 else "Light selling",
              tone(latest_conf_down < 0.3))),

        ("Weak / Unconfirmed Moves",
         f"{latest_weak:.1%}",
         pill("Low conviction" if latest_weak > 0.5 else "Conviction present",
              "neutral" if 0.4 < latest_weak < 0.6
              else tone(latest_weak < 0.5))),
    ]

    html = f"""
    <div style="overflow-x:auto; margin:20px 0; font-family:'Georgia',serif;">

    <!-- Header banner -->
    <div style="background:{C['burgundy']}; padding:14px 20px;
                border-radius:6px 6px 0 0; display:flex;
                justify-content:space-between; align-items:center;">
      <span style="color:{C['gold_light']}; font-size:15px; font-weight:700;
                   letter-spacing:0.08em;">
        ◈ &nbsp; VOLUME SIGNAL — {ticker.upper()}
      </span>
      <span style="color:{C['brown_lt']}; font-size:11px; font-style:italic;">
        {df.index[0].date()} &nbsp;→&nbsp; {df.index[-1].date()}
      </span>
    </div>

    <!-- Table -->
    <table style="border-collapse:collapse; font-size:12px;
                  width:100%; background:{C['bg']};
                  border:1px solid {C['border']};">
    <thead><tr>
      <th style="background:{C['bg_dark']}; color:{C['brown']};
                 padding:9px 16px; text-align:left; font-size:11px;
                 letter-spacing:0.06em; border-bottom:2px solid {C['gold']};
                 font-family:'Georgia',serif;">INDICATOR</th>
      <th style="background:{C['bg_dark']}; color:{C['brown']};
                 padding:9px 16px; text-align:right; font-size:11px;
                 letter-spacing:0.06em; border-bottom:2px solid {C['gold']};
                 font-family:'Georgia',serif;">VALUE</th>
      <th style="background:{C['bg_dark']}; color:{C['brown']};
                 padding:9px 16px; text-align:center; font-size:11px;
                 letter-spacing:0.06em; border-bottom:2px solid {C['gold']};
                 font-family:'Georgia',serif;">SIGNAL</th>
    </tr></thead><tbody>"""

    for i, (label, value, signal) in enumerate(rows):
        bg = C["bg"] if i % 2 == 0 else C["bg_dark"]
        html += (
            f'<tr style="background:{bg};">'
            f'<td style="padding:9px 16px; border-bottom:1px solid {C["border"]};'
            f'    color:{C["brown"]}; font-family:Georgia,serif;">{label}</td>'
            f'<td style="padding:9px 16px; border-bottom:1px solid {C["border"]};'
            f'    text-align:right; font-weight:600; color:{C["brown"]};'
            f'    font-family:Georgia,serif; letter-spacing:0.02em;">{value}</td>'
            f'<td style="padding:9px 16px; border-bottom:1px solid {C["border"]};'
            f'    text-align:center;">{signal}</td>'
            f'</tr>'
        )

    # Score bar
    bar_w     = min(score, 100)
    bar_color = (C["sage"] if score >= 50
                 else C["gold"] if score >= 35
                 else C["burgundy"])

    html += f"""
    </tbody></table>

    <!-- Conviction footer -->
    <div style="background:{conv_bg}; border:1px solid {C['border']};
                border-top:none; padding:14px 20px;
                border-radius:0 0 6px 6px;">

      <!-- Score bar -->
      <div style="margin-bottom:10px;">
        <div style="display:flex; justify-content:space-between;
                    margin-bottom:4px;">
          <span style="font-size:11px; color:{C['brown_mid']};
                       font-family:Georgia,serif;
                       letter-spacing:0.05em;">CONVICTION SCORE</span>
          <span style="font-size:11px; font-weight:700;
                       color:{conv_color};">{score} / 100</span>
        </div>
        <div style="background:{C['border']}; border-radius:3px; height:6px; width:100%;">
          <div style="background:{bar_color}; width:{bar_w}%;
                      height:6px; border-radius:3px;
                      transition:width 0.3s;"></div>
        </div>
      </div>

      <!-- Verdict -->
      <div style="display:flex; justify-content:space-between; align-items:center;">
        <span style="font-size:13px; font-weight:700;
                     color:{conv_color}; font-family:Georgia,serif;
                     letter-spacing:0.06em;">{conviction}</span>
        <span style="font-size:12px; color:{C['brown_mid']};
                     font-style:italic; font-family:Georgia,serif;">{verdict}</span>
      </div>
    </div>
    </div>"""

    display(HTML(html))

    # ─────────────────────────────────────────────
    # 6. Dashboard — warm finance visual style
    # ─────────────────────────────────────────────
    if plot:
        n      = 252
        dates  = df.index[-n:]
        c_     = close.iloc[-n:]
        v_     = volume.iloc[-n:]
        obv_   = obv.iloc[-n:]
        obv_s_ = obv_signal.iloc[-n:]
        ad_    = ad_line.iloc[-n:]
        cmf_   = cmf.iloc[-n:]
        vwap_  = rolling_vwap.iloc[-n:]
        lr_v_  = long_run_vwap.iloc[-n:]
        vma_s_ = vol_ma_short.iloc[-n:]
        vma_l_ = vol_ma_long.iloc[-n:]
        vol_r_ = volume_ratio.iloc[-n:]

        # Set global matplotlib style for warm theme
        plt.rcParams.update({
            "font.family":       "serif",
            "axes.facecolor":    C["bg"],
            "figure.facecolor":  C["bg_dark"],
            "axes.edgecolor":    C["border"],
            "axes.labelcolor":   C["brown"],
            "xtick.color":       C["brown_mid"],
            "ytick.color":       C["brown_mid"],
            "text.color":        C["brown"],
            "grid.color":        C["border"],
            "grid.alpha":        0.5,
        })

        fig = plt.figure(figsize=(16, 15))
        fig.patch.set_facecolor(C["bg_dark"])

        gs = gridspec.GridSpec(
            4, 2, figure=fig,
            hspace=0.6, wspace=0.35,
            height_ratios=[2.2, 1.3, 1.3, 1.3]
        )

        ax1 = fig.add_subplot(gs[0, :])
        ax2 = fig.add_subplot(gs[1, 0])
        ax3 = fig.add_subplot(gs[1, 1])
        ax4 = fig.add_subplot(gs[2, 0])
        ax5 = fig.add_subplot(gs[2, 1])
        ax6 = fig.add_subplot(gs[3, :])

        date_fmt = plt.matplotlib.dates.DateFormatter("%b '%y")
        def dress(ax, title):
            ax.spines[["top", "right"]].set_visible(False)
            ax.spines["left"].set_color(C["border"])
            ax.spines["bottom"].set_color(C["border"])
            ax.set_facecolor(C["bg"])
            ax.tick_params(labelsize=8, colors=C["brown_mid"])
            ax.set_title(title, fontsize=10, fontweight="bold",
                         color=C["brown"], pad=8,
                         fontfamily="serif")
            ax.xaxis.set_major_formatter(date_fmt)
            ax.xaxis.set_major_locator(
                plt.matplotlib.dates.MonthLocator(interval=2))
            plt.setp(ax.xaxis.get_majorticklabels(),
                     rotation=30, ha="right", fontsize=8)
            ax.grid(axis="y", linestyle="--", alpha=0.4,
                    color=C["border"])

        def fmt_millions(ax):
            ax.yaxis.set_major_formatter(
                plt.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M"))

        def fmt_price(ax):
            ax.yaxis.set_major_formatter(
                plt.FuncFormatter(lambda v, _: f"{v:,.1f}"))

        # ── Panel 1: Price + VWAP (full width) ──────────
        ax1.plot(dates, c_,     color=C["brown"],    linewidth=1.8,
                 label=f"Price  {latest_close:,.2f}", zorder=5)
        ax1.plot(dates, vwap_,  color=C["gold"],     linewidth=1.5,
                 linestyle="--",
                 label=f"VWAP({vwap_window}d)  {latest_vwap:,.2f}")
        ax1.plot(dates, lr_v_,  color=C["burgundy"], linewidth=1.2,
                 linestyle=":",
                 label=f"LR-VWAP  {latest_lr_vwap:,.2f}", alpha=0.8)
        ax1.fill_between(dates, c_, vwap_,
                         where=(c_ >= vwap_),
                         alpha=0.12, color=C["sage"],
                         label="Price above VWAP")
        ax1.fill_between(dates, c_, vwap_,
                         where=(c_ < vwap_),
                         alpha=0.12, color=C["burgundy"],
                         label="Price below VWAP")
        ax1.legend(fontsize=8, loc="upper left", framealpha=0.85,
                   facecolor=C["bg"], edgecolor=C["border"])
        fmt_price(ax1)
        dress(ax1, f"Price vs VWAP — {ticker.upper()}")

        # ── Panel 2: OBV ─────────────────────────────────
        ax2.plot(dates, obv_,   color=C["gold"],    linewidth=1.6,
                 label="OBV", zorder=4)
        ax2.plot(dates, obv_s_, color=C["burgundy"], linewidth=1.2,
                 linestyle="--",
                 label=f"Signal ({obv_window}d MA)", alpha=0.85)
        ax2.fill_between(dates, obv_, obv_s_,
                         where=(obv_ >= obv_s_),
                         alpha=0.15, color=C["sage"])
        ax2.fill_between(dates, obv_, obv_s_,
                         where=(obv_ <  obv_s_),
                         alpha=0.15, color=C["burgundy"])
        ax2.legend(fontsize=8, framealpha=0.85,
                   facecolor=C["bg"], edgecolor=C["border"])
        fmt_millions(ax2)
        dress(ax2, "On-Balance Volume (OBV)")

        # ── Panel 3: A/D Line ─────────────────────────────
        ax3.plot(dates, ad_,    color=C["gold"],    linewidth=1.6,
                 label="A/D Line")
        ax3.fill_between(dates, ad_, ad_.iloc[0],
                         where=(ad_ >= ad_.iloc[0]),
                         alpha=0.15, color=C["sage"])
        ax3.fill_between(dates, ad_, ad_.iloc[0],
                         where=(ad_ <  ad_.iloc[0]),
                         alpha=0.15, color=C["burgundy"])
        ax3.axhline(float(ad_.iloc[0]), color=C["brown_lt"],
                    linewidth=0.7, linestyle="--", alpha=0.6)
        ax3.legend(fontsize=8, framealpha=0.85,
                   facecolor=C["bg"], edgecolor=C["border"])
        fmt_millions(ax3)
        dress(ax3, "Accumulation / Distribution Line")

        # ── Panel 4: CMF ──────────────────────────────────
        ax4.bar(dates, cmf_.where(cmf_ >= 0, 0),
                color=C["sage"],    alpha=0.8, width=1.5,
                label="Accumulation")
        ax4.bar(dates, cmf_.where(cmf_ < 0, 0),
                color=C["burgundy"], alpha=0.8, width=1.5,
                label="Distribution")
        ax4.axhline( 0.15, color=C["gold"],    linewidth=1.0,
                    linestyle="--", alpha=0.7, label="+0.15")
        ax4.axhline(-0.15, color=C["burgundy"], linewidth=1.0,
                    linestyle="--", alpha=0.7, label="−0.15")
        ax4.axhline(0,     color=C["brown_lt"], linewidth=0.7, alpha=0.5)
        ax4.set_ylim(-1, 1)
        ax4.legend(fontsize=7, framealpha=0.85, ncol=2,
                   facecolor=C["bg"], edgecolor=C["border"])
        dress(ax4,
              f"Chaikin Money Flow ({cmf_window}d)"
              f"  ·  current: {latest_cmf:+.4f}")

        # ── Panel 5: Volume bars + MAs ────────────────────
        vol_colors = [
            C["sage"] if float(close.iloc[-(n-i)]) >= float(close.iloc[-(n-i)-1])
            else C["burgundy"]
            for i in range(len(v_))
        ]
        ax5.bar(dates, v_, color=vol_colors, alpha=0.55, width=1.2)
        ax5.plot(dates, vma_s_, color=C["gold"],    linewidth=1.5,
                 label=f"MA{vol_short}")
        ax5.plot(dates, vma_l_, color=C["brown_mid"], linewidth=1.2,
                 linestyle="--", label=f"MA{vol_long}")
        ax5.legend(fontsize=8, framealpha=0.85,
                   facecolor=C["bg"], edgecolor=C["border"])
        fmt_millions(ax5)
        dress(ax5, "Volume + Moving Averages")

        # ── Panel 6: Volume Ratio (full width) ───────────
        ax6.plot(dates, vol_r_, color=C["gold"],    linewidth=1.8,
                 label=f"Volume Ratio  (MA{vol_short} / MA{vol_long})")
        ax6.axhline(1.0,  color=C["brown_mid"], linewidth=0.9,
                    linestyle="--", alpha=0.7, label="Neutral (1.0)")
        ax6.axhline(1.15, color=C["sage"],    linewidth=0.8,
                    linestyle=":",  alpha=0.7, label="High (1.15)")
        ax6.axhline(0.85, color=C["burgundy"], linewidth=0.8,
                    linestyle=":",  alpha=0.7, label="Low (0.85)")
        ax6.fill_between(dates, vol_r_, 1.0,
                         where=(vol_r_ >= 1.0),
                         alpha=0.12, color=C["sage"])
        ax6.fill_between(dates, vol_r_, 1.0,
                         where=(vol_r_ <  1.0),
                         alpha=0.12, color=C["burgundy"])
        # Annotate current ratio
        ax6.annotate(
            f"  Current: {latest_vol_ratio:.2f}×",
            xy=(dates[-1], latest_vol_ratio),
            fontsize=9, color=C["gold"], fontweight="bold",
            fontfamily="serif",
        )
        ax6.legend(fontsize=8, framealpha=0.85, ncol=4,
                   facecolor=C["bg"], edgecolor=C["border"])
        ax6.yaxis.set_major_formatter(
            plt.FuncFormatter(lambda v, _: f"{v:.2f}×"))
        dress(ax6,
              f"Volume Activity Ratio  ·  current: {latest_vol_ratio:.2f}×")

        # ── Main title ────────────────────────────────────
        fig.suptitle(
            f"VolumeSignal  ◈  {ticker.upper()}  ◈  "
            f"Conviction: {score}/100  ◈  {conviction}",
            fontsize=13, fontweight="bold",
            color=C["gold"], y=1.01,
            fontfamily="serif",
        )

        plt.tight_layout()
        plt.show()

    return None


# ─────────────────────────────────────────────
# Example calls
# ─────────────────────────────────────────────
# volume_signal("ABUK.CA")
# volume_signal("AAPL")
# volume_signal("COMI.CA",  start="2018-01-01")
# volume_signal("ABUK.CA",  cmf_window=14, vol_short=10, vol_long=30)
# volume_signal("ABUK.CA",  plot=False)          # table only


In [7]:
tickers = ["ELWA.CA","SPMD.CA","ETEL.CA","GE","MSFT","ACAMD.CA","MAAL.CA","JUFO.CA","GM","GBCO.CA","CMI","ARAB.CA","CCAP.CA","ARVA.CA","AIFI.CA","UEGC.CA","AJWA.CA","AMES.CA","ELEC.CA","KO"]
for t in tickers:
    volume_signal(t , plot=False)

INDICATOR,VALUE,SIGNAL
Current Price,2.03,vs VWAP +6.88%
On-Balance Volume,"1,184,727,165",Rising ↑
Chaikin Money Flow,+0.4000,Strong accumulation
A/D Line Trend,Accumulating,Accumulation ↑
Volume Activity,Ratio 0.75×,Low ↓↓
Rolling VWAP,1.90,Above VWAP ↑
Long-Run VWAP,0.98,Above ↑
Confirmed Up Days (20D),30.0%,Weak buying
Confirmed Down Days (20D),5.0%,Light selling
Weak / Unconfirmed Moves,65.0%,Low conviction


INDICATOR,VALUE,SIGNAL
Current Price,0.41,vs VWAP +3.74%
On-Balance Volume,"3,824,714,403",Rising ↑
Chaikin Money Flow,-0.2248,Strong distribution
A/D Line Trend,Accumulating,Accumulation ↑
Volume Activity,Ratio 1.38×,High ↑↑
Rolling VWAP,0.40,Above VWAP ↑
Long-Run VWAP,0.61,Below ↓
Confirmed Up Days (20D),30.0%,Weak buying
Confirmed Down Days (20D),5.0%,Light selling
Weak / Unconfirmed Moves,65.0%,Low conviction


INDICATOR,VALUE,SIGNAL
Current Price,97.63,vs VWAP +1.42%
On-Balance Volume,"609,416,468",Rising ↑
Chaikin Money Flow,+0.0366,Mild accumulation
A/D Line Trend,Accumulating,Accumulation ↑
Volume Activity,Ratio 0.96×,Below average ↓
Rolling VWAP,96.26,Above VWAP ↑
Long-Run VWAP,17.83,Above ↑
Confirmed Up Days (20D),10.0%,Weak buying
Confirmed Down Days (20D),5.0%,Light selling
Weak / Unconfirmed Moves,80.0%,Low conviction


INDICATOR,VALUE,SIGNAL
Current Price,324.60,vs VWAP +7.75%
On-Balance Volume,"-500,515,880",Rising ↑
Chaikin Money Flow,+0.1467,Mild accumulation
A/D Line Trend,Accumulating,Accumulation ↑
Volume Activity,Ratio 0.87×,Below average ↓
Rolling VWAP,301.25,Above VWAP ↑
Long-Run VWAP,81.32,Above ↑
Confirmed Up Days (20D),10.0%,Weak buying
Confirmed Down Days (20D),0.0%,Light selling
Weak / Unconfirmed Moves,90.0%,Low conviction


INDICATOR,VALUE,SIGNAL
Current Price,460.52,vs VWAP +8.91%
On-Balance Volume,"2,962,132,475",Falling ↓
Chaikin Money Flow,+0.1091,Mild accumulation
A/D Line Trend,Accumulating,Accumulation ↑
Volume Activity,Ratio 1.01×,Above average ↑
Rolling VWAP,422.85,Above VWAP ↑
Long-Run VWAP,219.71,Above ↑
Confirmed Up Days (20D),20.0%,Weak buying
Confirmed Down Days (20D),5.0%,Light selling
Weak / Unconfirmed Moves,75.0%,Low conviction


...

INDICATOR,VALUE,SIGNAL
Current Price,1.32,vs VWAP -3.94%
On-Balance Volume,"2,853,529,017",Falling ↓
Chaikin Money Flow,-0.3291,Strong distribution
A/D Line Trend,Distributing,Distribution ↓
Volume Activity,Ratio 0.90×,Below average ↓
Rolling VWAP,1.37,Below VWAP ↓
Long-Run VWAP,0.91,Above ↑
Confirmed Up Days (20D),15.0%,Weak buying
Confirmed Down Days (20D),10.0%,Light selling
Weak / Unconfirmed Moves,70.0%,Low conviction


INDICATOR,VALUE,SIGNAL
Current Price,132.92,vs VWAP -0.69%
On-Balance Volume,"90,599,053",Falling ↓
Chaikin Money Flow,-0.1392,Mild distribution
A/D Line Trend,Distributing,Distribution ↓
Volume Activity,Ratio 0.76×,Low ↓↓
Rolling VWAP,133.84,Below VWAP ↓
Long-Run VWAP,27.40,Above ↑
Confirmed Up Days (20D),20.0%,Weak buying
Confirmed Down Days (20D),0.0%,Light selling
Weak / Unconfirmed Moves,80.0%,Low conviction


INDICATOR,VALUE,SIGNAL
Current Price,52.30,vs VWAP -7.58%
On-Balance Volume,"16,996,816",Falling ↓
Chaikin Money Flow,-0.0094,Mild distribution
A/D Line Trend,Distributing,Distribution ↓
Volume Activity,Ratio 0.93×,Below average ↓
Rolling VWAP,56.59,Below VWAP ↓
Long-Run VWAP,35.46,Above ↑
Confirmed Up Days (20D),15.0%,Weak buying
Confirmed Down Days (20D),5.0%,Light selling
Weak / Unconfirmed Moves,80.0%,Low conviction


INDICATOR,VALUE,SIGNAL
Current Price,2.11,vs VWAP -2.91%
On-Balance Volume,"4,136,522,498",Falling ↓
Chaikin Money Flow,-0.4378,Strong distribution
A/D Line Trend,Distributing,Distribution ↓
Volume Activity,Ratio 1.03×,Above average ↑
Rolling VWAP,2.17,Below VWAP ↓
Long-Run VWAP,0.76,Above ↑
Confirmed Up Days (20D),10.0%,Weak buying
Confirmed Down Days (20D),20.0%,Light selling
Weak / Unconfirmed Moves,70.0%,Low conviction


INDICATOR,VALUE,SIGNAL
Current Price,78.64,vs VWAP -1.70%
On-Balance Volume,"1,638,798,377",Falling ↓
Chaikin Money Flow,-0.1509,Strong distribution
A/D Line Trend,Distributing,Distribution ↓
Volume Activity,Ratio 0.99×,Below average ↓
Rolling VWAP,80.00,Below VWAP ↓
Long-Run VWAP,48.90,Above ↑
Confirmed Up Days (20D),20.0%,Weak buying
Confirmed Down Days (20D),25.0%,Light selling
Weak / Unconfirmed Moves,55.0%,Low conviction
